In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import lightgbm as lgb
import joblib
import time

In [3]:
df90 = pd.read_csv("../../datasets/df_pca_90.csv")
df95 = pd.read_csv("../../datasets/df_pca_95.csv")

In [4]:
X90 = df90.drop(columns=['label'])
y90 = df90['label']
X95 = df95.drop(columns=['label'])
y95 = df95['label']

In [5]:
print("\nTraining Decision Tree(90%)...")
start = time.time()
dt90 = DecisionTreeClassifier(max_depth=10, random_state=42)
dt90.fit(X90, y90)
y_pred_dt90 = dt90.predict(X90)
end = time.time()
print(f"Decision Tree Training(90%) Time: {end - start:.2f} sec")


Training Decision Tree(90%)...
Decision Tree Training(90%) Time: 377.85 sec


In [6]:
print("\nDecision Tree Report(90%):")
print("Accuracy:", accuracy_score(y90, y_pred_dt90))
print(classification_report(y90, y_pred_dt90))
joblib.dump(dt90, "./pcamodels/decision_tree_model_with_no_feature_filtration_90.pkl")


Decision Tree Report(90%):
Accuracy: 0.8784983333333334
              precision    recall  f1-score   support

           0       0.89      0.89      0.89   2696949
           1       0.86      0.86      0.86   2103051

    accuracy                           0.88   4800000
   macro avg       0.88      0.88      0.88   4800000
weighted avg       0.88      0.88      0.88   4800000



['./pcamodels/decision_tree_model_with_no_feature_filtration_90.pkl']

In [7]:
print("\nTraining Decision Tree(95%)...")
start = time.time()
dt95 = DecisionTreeClassifier(max_depth=10, random_state=42)
dt95.fit(X95, y95)
y_pred_dt95 = dt95.predict(X95)
end = time.time()
print(f"Decision Tree Training(95%) Time: {end - start:.2f} sec")


Training Decision Tree(95%)...
Decision Tree Training(95%) Time: 605.85 sec


In [8]:
print("\nDecision Tree Report(95%):")
print("Accuracy:", accuracy_score(y95, y_pred_dt95))
print(classification_report(y95, y_pred_dt95))
joblib.dump(dt95, "./pcamodels/decision_tree_model_with_no_feature_filtration_95.pkl")


Decision Tree Report(95%):
Accuracy: 0.884554375
              precision    recall  f1-score   support

           0       0.91      0.88      0.90   2696949
           1       0.86      0.89      0.87   2103051

    accuracy                           0.88   4800000
   macro avg       0.88      0.88      0.88   4800000
weighted avg       0.89      0.88      0.88   4800000



['./pcamodels/decision_tree_model_with_no_feature_filtration_95.pkl']

In [9]:
train_data = lgb.Dataset(X90, label=y90)

params = {
    "objective": "binary",  
    "boosting": "gbdt",
    "metric": "binary_error", 
    "num_leaves": 64,
    "learning_rate": 0.1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1
}

print("\nTraining LightGBM on full df_pca_90.csv ...")
start = time.time()

# Train model (no validation split)
model = lgb.train(
    params,
    train_data,
    num_boost_round=500
)

end = time.time()
print(f"✅ LightGBM Training Time(90%): {end - start:.2f} sec")

# Predictions on training data
y_pred90 = model.predict(X90)
y_pred_binary90 = (y_pred90 > 0.5).astype(int)


Training LightGBM on full df_pca_90.csv ...
✅ LightGBM Training Time(90%): 214.77 sec


In [10]:
print("\nLightGBM Report (Train Set)(90%):")
print("Accuracy:", accuracy_score(y90, y_pred_binary90))
print(classification_report(y90, y_pred_binary90))
model.save_model("./pcamodels/lightgbm_model_90.txt")



LightGBM Report (Train Set)(90%):
Accuracy: 0.9244222916666667
              precision    recall  f1-score   support

           0       0.94      0.93      0.93   2696949
           1       0.91      0.92      0.91   2103051

    accuracy                           0.92   4800000
   macro avg       0.92      0.92      0.92   4800000
weighted avg       0.92      0.92      0.92   4800000



In [11]:
train_data = lgb.Dataset(X95, label=y95)

params = {
    "objective": "binary",  
    "boosting": "gbdt",
    "metric": "binary_error", 
    "num_leaves": 64,
    "learning_rate": 0.1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1
}

print("\nTraining LightGBM on full df_pca_95.csv ...")
start = time.time()

# Train model (no validation split)
model = lgb.train(
    params,
    train_data,
    num_boost_round=500
)

end = time.time()
print(f"✅ LightGBM Training Time(95%): {end - start:.2f} sec")

# Predictions on training data
y_pred95 = model.predict(X95)
y_pred_binary95 = (y_pred95 > 0.5).astype(int)


Training LightGBM on full df_pca_95.csv ...
✅ LightGBM Training Time(95%): 163.97 sec


In [12]:
print("\nLightGBM Report (Train Set)(95%):")
print("Accuracy:", accuracy_score(y95, y_pred_binary95))
print(classification_report(y95, y_pred_binary95))
model.save_model("./pcamodels/lightgbm_model_95.txt")


LightGBM Report (Train Set)(95%):
Accuracy: 0.92766125
              precision    recall  f1-score   support

           0       0.94      0.93      0.94   2696949
           1       0.91      0.92      0.92   2103051

    accuracy                           0.93   4800000
   macro avg       0.93      0.93      0.93   4800000
weighted avg       0.93      0.93      0.93   4800000



In [13]:
#90%
nb90 = GaussianNB()
nb90.fit(X90, y90)

y_pred90 = nb90.predict(X90)

print("Accuracy:", accuracy_score(y90, y_pred90))
print(classification_report(y90, y_pred90))

#95%
nb95 = GaussianNB()
nb95.fit(X95, y95)

y_pred95 = nb95.predict(X95)

print("Accuracy:", accuracy_score(y95, y_pred95))
print(classification_report(y95, y_pred95))

Accuracy: 0.6882152083333334
              precision    recall  f1-score   support

           0       0.65      0.95      0.77   2696949
           1       0.84      0.35      0.50   2103051

    accuracy                           0.69   4800000
   macro avg       0.75      0.65      0.64   4800000
weighted avg       0.74      0.69      0.65   4800000

Accuracy: 0.6714645833333334
              precision    recall  f1-score   support

           0       0.64      0.94      0.76   2696949
           1       0.81      0.33      0.46   2103051

    accuracy                           0.67   4800000
   macro avg       0.73      0.63      0.61   4800000
weighted avg       0.72      0.67      0.63   4800000



In [14]:
print("\nTraining XGBoost (baseline params)(90%)...")
start = time.time()

# Baseline XGBoost model
xgb90 = XGBClassifier(
    n_estimators=200,       # number of trees (baseline)
    max_depth=3,            # depth of trees
    objective='binary:logistic',  # binary classification
    n_jobs=-1,
    random_state=42
)

xgb90.fit(X90, y90)

end = time.time()
print(f"XGBoost Training Time(90%): {end - start:.2f} sec")

# Predictions on train set
y_pred90 = xgb90.predict(X90)


Training XGBoost (baseline params)(90%)...
XGBoost Training Time(90%): 43.96 sec


In [15]:
print("\nXGBoost Report (Train Set)(90%):")
print("Accuracy:", accuracy_score(y90, y_pred90))
print(classification_report(y90, y_pred90))
xgb90.save_model("./pcamodels/xgboost_90.json")


XGBoost Report (Train Set)(90%):
Accuracy: 0.89824375
              precision    recall  f1-score   support

           0       0.92      0.90      0.91   2696949
           1       0.87      0.90      0.89   2103051

    accuracy                           0.90   4800000
   macro avg       0.90      0.90      0.90   4800000
weighted avg       0.90      0.90      0.90   4800000



In [17]:
print("\nTraining XGBoost (baseline params)(95%)...")
start = time.time()

# Baseline XGBoost model
xgb95 = XGBClassifier(
    n_estimators=200,       # number of trees (baseline)
    max_depth=3,            # depth of trees
    objective='binary:logistic',  # binary classification
    n_jobs=-1,
    random_state=42
)

xgb95.fit(X95, y95)

end = time.time()
print(f"XGBoost Training Time(95%): {end - start:.2f} sec")
y_pred95 = xgb95.predict(X95)


Training XGBoost (baseline params)(95%)...
XGBoost Training Time(95%): 69.57 sec


In [ ]:
print("\nXGBoost Report (Train Set)(90%):")
print("Accuracy:", accuracy_score(y95, y_pred95))
print(classification_report(y95, y_pred95))
xgb95.save_model("./pcamodels/xgboost_95.json")


XGBoost Report (Train Set)(90%):
Accuracy: 0.904749375
              precision    recall  f1-score   support

           0       0.92      0.91      0.91   2696949
           1       0.88      0.90      0.89   2103051

    accuracy                           0.90   4800000
   macro avg       0.90      0.90      0.90   4800000
weighted avg       0.91      0.90      0.90   4800000

